# Iris Classification - Supervised Learning with Multiple Classifiers

This notebook demonstrates supervised learning using the Iris dataset with multiple classification algorithms.

## Objectives:
1. Load and explore the Iris dataset
2. Preprocess and visualize the data
3. Train multiple classifiers (Logistic Regression, Decision Tree, Random Forest, SVM)
4. Evaluate models using various metrics
5. Perform hyperparameter tuning and cross-validation
6. Compare model performance

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve, auc)
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")

## 1. Load and Explore the Iris Dataset

In [ ]:
# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"\nFeature names: {feature_names}")
print(f"Target classes: {target_names}")

In [ ]:
# Create a DataFrame for better visualization
df = pd.DataFrame(X, columns=feature_names)
df['Species'] = [target_names[i] for i in y]

print("Dataset Overview:")
print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Species'].value_counts())

In [ ]:
# Statistical summary
print(df.describe())

## 2. Data Visualization

In [ ]:
# Visualization: Scatter plots for feature relationships
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sepal Length vs Sepal Width
for i, species in enumerate(target_names):
    mask = y == i
    axes[0, 0].scatter(X[mask, 0], X[mask, 1], label=species, alpha=0.7, s=60)
axes[0, 0].set_xlabel('Sepal Length (cm)')
axes[0, 0].set_ylabel('Sepal Width (cm)')
axes[0, 0].set_title('Sepal Length vs Sepal Width')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Petal Length vs Petal Width
for i, species in enumerate(target_names):
    mask = y == i
    axes[0, 1].scatter(X[mask, 2], X[mask, 3], label=species, alpha=0.7, s=60)
axes[0, 1].set_xlabel('Petal Length (cm)')
axes[0, 1].set_ylabel('Petal Width (cm)')
axes[0, 1].set_title('Petal Length vs Petal Width (Best Separable)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Sepal Length vs Petal Length
for i, species in enumerate(target_names):
    mask = y == i
    axes[1, 0].scatter(X[mask, 0], X[mask, 2], label=species, alpha=0.7, s=60)
axes[1, 0].set_xlabel('Sepal Length (cm)')
axes[1, 0].set_ylabel('Petal Length (cm)')
axes[1, 0].set_title('Sepal Length vs Petal Length')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Feature distributions
for j, species in enumerate(target_names):
    mask = y == j
    axes[1, 1].hist(X[mask, 2], alpha=0.5, label=species, bins=15)
axes[1, 1].set_xlabel('Petal Length (cm)')
axes[1, 1].set_title('Petal Length Distribution by Species')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Train-test split (80/20)
random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

print(f"Training set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nClass distribution in training set: {np.bincount(y_train)}")
print(f"Class distribution in test set: {np.bincount(y_test)}")

In [ ]:
# Feature normalization using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features normalized (mean=0, std=1)")
print(f"Mean of scaled features: {X_train_scaled.mean(axis=0)}")
print(f"Std of scaled features: {X_train_scaled.std(axis=0)}")

## 4. Model Training

In [ ]:
# Initialize models
models = {}

# 1. Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=200, random_state=random_state)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr
print("✓ Trained")

# 2. Decision Tree
print("Training Decision Tree...")
dt = DecisionTreeClassifier(random_state=random_state, max_depth=5)
dt.fit(X_train, y_train)
models['Decision Tree'] = dt
print("✓ Trained")

# 3. Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=random_state, max_depth=5)
rf.fit(X_train, y_train)
models['Random Forest'] = rf
print("✓ Trained")

# 4. SVM
print("Training SVM...")
svm = SVC(kernel='rbf', random_state=random_state, probability=True)
svm.fit(X_train_scaled, y_train)
models['SVM'] = svm
print("✓ Trained")

print("\nAll models trained successfully!")

## 5. Model Evaluation

In [ ]:
# Evaluate all models
results = {}

for model_name, model in models.items():
    # Make predictions
    if model_name in ['Logistic Regression', 'SVM']:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'y_pred': y_pred,
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }

    print(f"{model_name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print()

## 6. Performance Visualization

In [ ]:
# Create performance comparison chart
metrics_data = {model: [results[model][metric]
                         for metric in ['accuracy', 'precision', 'recall', 'f1']]
                for model in models.keys()}

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(models))
width = 0.2

for i, metric in enumerate(['accuracy', 'precision', 'recall', 'f1']):
    values = [results[model][metric] for model in models.keys()]
    ax.bar(x + i*width, values, width, label=metric.capitalize())

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(models.keys(), rotation=15, ha='right')
ax.legend()
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (model_name, model) in enumerate(models.items()):
    cm = results[model_name]['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
               xticklabels=target_names, yticklabels=target_names, cbar=True)
    axes[idx].set_title(f'{model_name} Confusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## 7. Classification Reports

In [ ]:
# Detailed classification reports
for model_name, model in models.items():
    if model_name in ['Logistic Regression', 'SVM']:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)

    print(f"\n{'='*80}")
    print(f"{model_name.upper()} - CLASSIFICATION REPORT")
    print(f"{'='*80}")
    print(classification_report(y_test, y_pred, target_names=target_names))

## 8. Hyperparameter Tuning

In [ ]:
# Random Forest hyperparameter tuning
print("Tuning Random Forest parameters...")
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=random_state)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5, n_jobs=-1, verbose=0)
grid_search_rf.fit(X_train, y_train)

print(f"Best parameters: {grid_search_rf.best_params_}")
print(f"Best CV score: {grid_search_rf.best_score_:.4f}")

best_rf = grid_search_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)
print(f"Test accuracy with best parameters: {accuracy_best_rf:.4f}")

In [ ]:
# SVM hyperparameter tuning
print("Tuning SVM parameters...")
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

svm = SVC(random_state=random_state, probability=True)
grid_search_svm = GridSearchCV(svm, param_grid_svm, cv=5, n_jobs=-1, verbose=0)
grid_search_svm.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search_svm.best_params_}")
print(f"Best CV score: {grid_search_svm.best_score_:.4f}")

best_svm = grid_search_svm.best_estimator_
y_pred_best_svm = best_svm.predict(X_test_scaled)
accuracy_best_svm = accuracy_score(y_test, y_pred_best_svm)
print(f"Test accuracy with best parameters: {accuracy_best_svm:.4f}")

## 9. Cross-Validation

In [ ]:
# 5-fold cross-validation for all models
print("5-Fold Cross-Validation Results:\n")

cv_results = {}
for model_name, model in models.items():
    if model_name in ['Logistic Regression', 'SVM']:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    cv_results[model_name] = cv_scores

    print(f"{model_name}:")
    print(f"  CV Scores: {[f'{x:.4f}' for x in cv_scores]}")
    print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print()

In [ ]:
# Visualize cross-validation results
fig, ax = plt.subplots(figsize=(10, 6))

positions = np.arange(len(cv_results))
cv_means = [cv_results[model].mean() for model in cv_results.keys()]
cv_stds = [cv_results[model].std() for model in cv_results.keys()]

ax.bar(positions, cv_means, yerr=cv_stds, capsize=5, alpha=0.7, color='steelblue')
ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Cross-Validation Score', fontsize=12)
ax.set_title('5-Fold Cross-Validation Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(positions)
ax.set_xticklabels(cv_results.keys(), rotation=15, ha='right')
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 10. Summary and Conclusions

In [ ]:
# Create summary table
summary_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1-Score': [results[m]['f1'] for m in results.keys()],
    'CV Mean': [cv_results[m].mean() for m in cv_results.keys()],
    'CV Std': [cv_results[m].std() for m in cv_results.keys()]
})

print("\n" + "="*100)
print("IRIS CLASSIFICATION - SUMMARY REPORT")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

# Identify best model
best_model = summary_df.loc[summary_df['Accuracy'].idxmax()]
print(f"\n✓ Best Performing Model: {best_model['Model']}")
print(f"  - Test Accuracy: {best_model['Accuracy']:.4f}")
print(f"  - CV Mean Score: {best_model['CV Mean']:.4f} (+/- {best_model['CV Std']:.4f})")

## Key Insights:

1. **Data Characteristics**: The Iris dataset has 4 features and 3 classes with good feature separation (especially petal measurements)

2. **Model Performance**: All models achieve >90% accuracy on the test set

3. **Algorithm Comparison**:
   - **Logistic Regression**: Fast, interpretable, good baseline
   - **Decision Tree**: Simple, prone to overfitting without pruning
   - **Random Forest**: Ensemble method, excellent performance, robust
   - **SVM**: Powerful non-linear classifier with RBF kernel

4. **Best Practices Applied**:
   - 80/20 train-test split with stratification
   - Feature scaling for distance-based algorithms
   - Multiple evaluation metrics (accuracy, precision, recall, F1)
   - Confusion matrices for detailed error analysis
   - Hyperparameter tuning using GridSearchCV
   - Cross-validation for robust performance estimation

5. **Recommendations**:
   - Use Random Forest for best generalization
   - Consider SVM for deployment if prediction speed matters
   - Use Logistic Regression for interpretability